In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from scipy.stats import genextreme, genpareto, pareto
from scipy.optimize import minimize

from wxderivs import SET_TARGET_US_CITIES, MONTH_CDD, MONTH_HDD, DIR_WORK

# Import seasonal modeling functions
from seasonal_models import (
    fit_seasonal_model,
    fit_seasonal_garch_model,
    compare_models,
    fit_all_regions,
    plot_model_comparison,
    plot_aic_bic_comparison,
    # Diagnostic functions
    plot_fitted_model_and_residuals,
    plot_residual_distribution,
    plot_autocorrelation,
    analyze_volatility_clustering,
    run_full_diagnostics
)

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

sys.path.append(str(DIR_WORK / "unibm"))

import unibm
import unibm.benchmark


0 /workspaces/Financial-Engineering-Project/notebooks/wxderivs/../../data/raw/temperature/Essen.csv
1 /workspaces/Financial-Engineering-Project/notebooks/wxderivs/../../data/raw/temperature/Boston.csv
2 /workspaces/Financial-Engineering-Project/notebooks/wxderivs/../../data/raw/temperature/Philadelphia.csv
3 /workspaces/Financial-Engineering-Project/notebooks/wxderivs/../../data/raw/temperature/NewYork.csv
4 /workspaces/Financial-Engineering-Project/notebooks/wxderivs/../../data/raw/temperature/Burbank.csv
5 /workspaces/Financial-Engineering-Project/notebooks/wxderivs/../../data/raw/temperature/London.csv
6 /workspaces/Financial-Engineering-Project/notebooks/wxderivs/../../data/raw/temperature/Dallas.csv
7 /workspaces/Financial-Engineering-Project/notebooks/wxderivs/../../data/raw/temperature/Paris.csv
8 /workspaces/Financial-Engineering-Project/notebooks/wxderivs/../../data/raw/temperature/Portland.csv
9 /workspaces/Financial-Engineering-Project/notebooks/wxderivs/../../data/raw/tempe

In [2]:
# Load Boston data (aligned columns: date, pred_min, pred_max, pred_avg, min_load, max_load, avg_load)
df_boston = pd.read_csv('../data/processed/Boston/Boston.csv')
df_boston['date'] = pd.to_datetime(df_boston['date'])
# df_boston = df_boston[pd.to_datetime(df_boston['date']).dt.dayofweek < 5]
df_boston = df_boston[(df_boston['date'] >= '2014-01-01') & (df_boston['date'] <= '2022-12-31')]
df_boston.set_index('date', inplace=True)

# Load other region data (aligned columns: date, min_load, max_load, avg_load)
df_ny = pd.read_csv('../data/processed/NY/NewYork.csv')

df_ny['date'] = pd.to_datetime(df_ny['date'])
# Remove weekends (Saturday=5, Sunday=6)
# df_ny = df_ny[pd.to_datetime(df_ny['date']).dt.dayofweek < 5]
df_ny = df_ny[(df_ny['date'] >= '2014-01-01') & (df_ny['date'] <= '2022-12-31')]
df_ny.set_index('date', inplace=True)

# df_houston = pd.read_csv('../data/processed/Houston/Houston.csv')
# df_houston =  df_houston[pd.to_datetime(df_houston['date']).dt.dayofweek < 5]
# df_houston['date'] = pd.to_datetime(df_houston['date'])
# df_houston = df_houston[(df_houston['date'] >= '2014-01-01') & (df_houston['date'] <= '2022-12-31')]
# df_houston.set_index('date', inplace=True)

df_chicago = pd.read_csv('../data/processed/Chicago/Chicago.csv')
df_chicago['date'] = pd.to_datetime(df_chicago['date'])
# df_chicago = df_chicago[pd.to_datetime(df_chicago['date']).dt.dayofweek < 5]
df_chicago = df_chicago[(df_chicago['date'] <= '2022-12-31') & (df_chicago['date'] >= '2014-01-01')]
df_chicago.set_index('date', inplace=True)

# df_dallas = pd.read_csv('../data/processed/Dallas/Dallas.csv')
# df_dallas['date'] = pd.to_datetime(df_dallas['date'])
# df_dallas = df_dallas[pd.to_datetime(df_dallas['date']).dt.dayofweek < 5]
# df_dallas = df_dallas[(df_dallas['date'] >= '2014-01-01') & (df_dallas['date'] <= '2022-12-31')]
# df_dallas.set_index('date', inplace=True)

df_minneapolis = pd.read_csv('../data/processed/Minneapolis/Minneapolis.csv')
df_minneapolis['date'] = pd.to_datetime(df_minneapolis['date'])
# df_minneapolis = df_minneapolis[pd.to_datetime(df_minneapolis['date']).dt.dayofweek < 5]
df_minneapolis = df_minneapolis[(df_minneapolis['date'] >= '2014-01-01') & (df_minneapolis['date'] <= '2022-12-31')]
df_minneapolis.set_index('date', inplace=True)

print("Boston data shape:", df_boston.shape)
print("New York data shape:", df_ny.shape)
print("Chicago data shape:", df_chicago.shape)
print("Minneapolis data shape:", df_minneapolis.shape)
print("\nBoston columns:", df_boston.columns.tolist())
print("\nBoston data range:", df_boston.index.min(), "to", df_boston.index.max())

Boston data shape: (3287, 6)
New York data shape: (3287, 3)
Chicago data shape: (3286, 3)
Minneapolis data shape: (3287, 3)

Boston columns: ['pred_min', 'pred_max', 'pred_avg', 'min_load', 'max_load', 'avg_load']

Boston data range: 2014-01-01 00:00:00 to 2022-12-31 00:00:00


In [ ]:
# Boston 3H + ARMA(0,2) + GARCH(1,2)
# New York 3H + ARMA(1,3) + GARCH(1,3)
# Chicago 3H + ARMA(1,3) + GARCH(2,3)
# Minneapolis 3H + ARMA(1,3) + GARCH(2,3)

print("Generating comprehensive visualizations for all regions...")

# Create dictionary of all region data
regions_to_plot = {
    'Boston': df_boston['avg_load'],
    'NewYork': df_ny['avg_load'],
    # 'Houston': df_houston['avg_load'],
    'Chicago': df_chicago['avg_load'],
    # 'Dallas': df_dallas['avg_load'],
    'Minneapolis': df_minneapolis['avg_load'],
}

region_fits = {}
for region_name, load_data in regions_to_plot.items():
    print(f"  Fitting {region_name}...")
    if region_name == 'Boston':
        region_fits[region_name] = fit_seasonal_garch_model(load_data, n_harmonics=3, ar_order=[0], ma_order=[2], garch_p_list=[1], garch_q_list=[2])
    elif region_name == 'NewYork':
        region_fits[region_name] = fit_seasonal_garch_model(load_data, n_harmonics=3, ar_order_list=[1], ma_order_list=[3], garch_p_list=[1], garch_q_list=[3])
    elif region_name == 'Chicago':
        region_fits[region_name] = fit_seasonal_garch_model(load_data, n_harmonics=3, ar_order_list=[1], ma_order_list=[3], garch_p_list=[2], garch_q_list=[3])
    elif region_name == 'Minneapolis':
        region_fits[region_name] = fit_seasonal_garch_model(load_data, n_harmonics=3, ar_order_list=[1], ma_order_list=[3], garch_p_list=[2], garch_q_list=[3])

# Create comprehensive visualization
fig, axes = plt.subplots(nrows=6, ncols=2, figsize=(18, 24))

for idx, (region, fit) in enumerate(region_fits.items()):
    # Left panel: Observed vs Fitted
    axes[idx, 0].scatter(fit['index'], fit['y'], s=0.5, alpha=0.4, 
                        color='blue', label='Observed')
    axes[idx, 0].plot(fit['index'], fit['y_pred'], 
                     color='red', lw=1.0, label='Fitted Mean', alpha=0.8)
    axes[idx, 0].set_ylabel('Load (MWh)', fontsize=10)
    axes[idx, 0].set_title(f'{region} - Observed vs Fitted', fontsize=11)
    axes[idx, 0].legend(fontsize=8)
    axes[idx, 0].grid(True, alpha=0.3)
    
    # Right panel: Residuals with GARCH bands
    axes[idx, 1].scatter(fit['index'], fit['residuals'], 
                        s=0.5, alpha=0.4, color='red', label='Residuals')
    axes[idx, 1].plot(fit['index'], fit['volatility'], 
                     color='darkgreen', lw=1.0, linestyle='--', label='+1σ')
    axes[idx, 1].plot(fit['index'], -fit['volatility'], 
                     color='darkgreen', lw=1.0, linestyle='--', label='-1σ')
    axes[idx, 1].axhline(0, color='black', linestyle='-', lw=0.8, alpha=0.5)
    axes[idx, 1].set_ylabel('Residuals (MWh)', fontsize=10)
    axes[idx, 1].set_title(f'{region} - Residuals with GARCH Bands', fontsize=11)
    axes[idx, 1].legend(fontsize=8)
    axes[idx, 1].grid(True, alpha=0.3)
    
    # Only show x-label on bottom row
    if idx == 5:
        axes[idx, 0].set_xlabel('Date', fontsize=10)
        axes[idx, 1].set_xlabel('Date', fontsize=10)

plt.suptitle('Seasonal (6H) + GARCH(1,1) Model Results - All Regions', fontsize=16, y=0.995)
plt.tight_layout()
plt.savefig('../data/processed/all_regions_seasonal_garch.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Visualization complete!")


Generating comprehensive visualizations for all regions...
  Fitting Boston...


TypeError: fit_seasonal_garch_model() got an unexpected keyword argument 'ar_order_list'